In [1]:
!pip install pandas

In [2]:
import pandas as pd
import numpy as np
import os
import json
import re
import copy

In [3]:
path_dados_modulo = '../../data'
path_csv = os.path.join(path_dados_modulo, 'csv/dados_devolutivas.csv')

assert os.path.exists(path_csv)

In [4]:
df = pd.read_csv(path_csv)

In [5]:
df.head()

,id_dev,id_cpma,id_participe_mais,canal_pad,tipo_origem,municipe_pad,titulo,resumo,conteudo_ajustado,tema_pad,orgao,devolutiva,apoios,total_comentarios,subprefeitura_pad
0,6123,420,3808.0,Participe Mais,proposta,Ricardo Souza,Segunda Entrada para o Jardim Keralux.,"Agilizar um segundo acesso ao jardim Keralux, ...",Minha proposta visa a criação de um segundo ac...,Obra viária,SMUL,O PIU Arco Leste tem propostas de melhoramento...,17.0,0.0,ERMELINO MATARAZZO
1,6650,6424,3809.0,Participe Mais,proposta,Diana Tsonis,Incorporar políticas federais ao plano de metas,Incorporar ao Plano de metas o Política Nacion...,Existem diversas políticas e planos federais q...,Cultura,SMC,A Secretaria Municipal de Cultura e Economia C...,34.0,0.0,NaN
2,8595,6426,3809.0,Participe Mais,proposta,Diana Tsonis,Incorporar políticas federais ao plano de metas,Incorporar ao Plano de metas o Política Nacion...,Existem diversas políticas e planos federais q...,Desenvolvimento econômico,SMDET,A Prefeitura de São Paulo agradece a contribui...,34.0,0.0,NaN
3,80,6425,3809.0,Participe Mais,proposta,Diana Tsonis,Incorporar políticas federais ao plano de metas,Incorporar ao Plano de metas o Política Nacion...,Existem diversas políticas e planos federais q...,Pessoa em situação de rua,SMDHC,"No que tange à SMDHC, avaliaremos a proposta n...",34.0,0.0,NaN
4,8142,146,3810.0,Participe Mais,proposta,Samuel Caetano Da Silva,Criação do Parque Linear Caxingui,A criação do Parque Linear Caxingui no Morumbi...,Contexto e Justificativa O distrito do Morumbi...,Parques,SVMA,"Não é viável a inserção neste momento, pois a ...",67.0,0.0,NaN


In [6]:
df.columns

Index(['id_dev', 'id_cpma', 'id_participe_mais', 'canal_pad', 'tipo_origem',
       'municipe_pad', 'titulo', 'resumo', 'conteudo_ajustado', 'tema_pad',
       'orgao', 'devolutiva', 'apoios', 'total_comentarios',
       'subprefeitura_pad'],
      dtype='object')

In [7]:
df['tipo_origem'].unique()

array(['proposta', 'revisao', 'fala'], dtype=object)

In [8]:
df['conteudo_ajustado'] = df['conteudo_ajustado'].str.strip()

In [9]:
df['id_contribuicao'] = np.nan

df['is_fala'] = df['tipo_origem']=='fala'

ids_falas = dict()
id_atual=0
remover = ['\n', '\t', '\r']
for i, row in df.iterrows():
    if row['is_fala']:
        fala = row['conteudo_ajustado'].strip()
        for r in remover:
            fala = fala.replace(r, '')
        #colocando a fala ajustada no df
        df.loc[i, 'conteudo_ajustado'] = fala
        if fala not in ids_falas:
            ids_falas[fala]=id_atual
            id_atual += 1
        df.loc[i, 'id_contribuicao'] = ids_falas[fala]
    else:
        df.loc[i, 'id_contribuicao'] = int(row['id_participe_mais'])

assert not df['id_contribuicao'].isnull().any()

df['id_contribuicao'] = df['tipo_origem'].str.slice(0, 3)+'_'+df['id_contribuicao'].astype(int).astype(str).str.zfill(5)


In [10]:
ids_falas[fala]

482

In [11]:
df[df['is_fala']].head()[['conteudo_ajustado', 'id_contribuicao']]

,conteudo_ajustado,id_contribuicao
1513,A gente convida a senhora Maria Lodi. Senhora ...,fal_00000
1514,A gente convida agora a participar a senhora A...,fal_00001
1515,A gente passa agora para a senhora Telmelita. ...,fal_00002
1516,A gente passa agora para a senhora Telmelita. ...,fal_00002
1517,A gente passa agora para a senhora Telmelita. ...,fal_00002


In [12]:
df.columns

Index(['id_dev', 'id_cpma', 'id_participe_mais', 'canal_pad', 'tipo_origem',
       'municipe_pad', 'titulo', 'resumo', 'conteudo_ajustado', 'tema_pad',
       'orgao', 'devolutiva', 'apoios', 'total_comentarios',
       'subprefeitura_pad', 'id_contribuicao', 'is_fala'],
      dtype='object')

In [13]:
df = df.replace('NA', None)

In [14]:
for col in df.columns:
    if df[col].dtype not in ('object', int):
        df[col] = df[col].fillna(0)
        df[col] = df[col].astype(int)

    if df[col].dtype == 'object':
        df[col] = df[col].replace(np.nan, None)
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace('None', None)


In [15]:
df.dtypes

id_dev                int64
id_cpma               int64
id_participe_mais     int64
canal_pad            object
tipo_origem          object
municipe_pad         object
titulo               object
resumo               object
conteudo_ajustado    object
tema_pad             object
orgao                object
devolutiva           object
apoios                int64
total_comentarios     int64
subprefeitura_pad    object
id_contribuicao      object
is_fala               int64
dtype: object

In [16]:
df['orgao'] = df['orgao'].replace('SPREGULA', 'SPRegula')
df['orgao'] = df['orgao'].replace('SPTURIS', 'SPTuris')
df['orgao'] = df['orgao'].replace('FUND. PAULISTANA', 'FPD')
df['orgao'] = df['orgao'].replace('PREF/CGA', 'SERI')
df['orgao'] = df['orgao'].replace('CASA CIVIL', 'CC')

df['orgao'] = df['orgao'].replace('None', None)

In [17]:
diagnostico = {

}

dados_parsed = []


def add_to_diagnostico(contrib, msg):
    if contrib not in diagnostico:
        diagnostico[contrib] = []
    diagnostico[contrib].append(msg)

def checar_qtds(df_contrib, mapper_colunas_atributos):

    for coluna, nome in mapper_colunas_atributos.items():
        qtd = df_contrib[coluna].nunique()
        if qtd > 1:
            unicos = df_contrib[coluna].unique()
            msg_especifica = f'Mais de um {nome}: {qtd}. {unicos}'
            add_to_diagnostico(contrib, msg_especifica)

def checar_not_null(df_contrib, coluna):
    if df_contrib[coluna].isnull().any():
        msg_especifica = f'Coluna {coluna} tem valor nulo'
        add_to_diagnostico(contrib, msg_especifica)
        return False
    return True


def extract_first_value(df, coluna, convert):

    return convert(df[coluna].iloc[0])

colunas_atributos_str = {
    'id_contribuicao' : 'id_contribuicao',
    'canal_pad' : 'canal',
    'tipo_origem' : 'tipo_origem',
    'municipe_pad' : 'municipe',
    'titulo' : 'titulo',
    'resumo' : 'resumo',
    'conteudo_ajustado' : 'conteudo',
}


colunas_atributos_int = {
    'id_participe_mais' : 'id_participe_mais',
    'apoios' : 'qtd_apoios',
    'total_comentarios' : 'qtd_comentarios',
}

colunas_atributos_many = {
    'subprefeitura_pad' : 'subprefeitura'
}


colunas_devolutiva = {
        'tema_pad' : 'tema',
        'orgao' : 'orgao',
        'devolutiva' : 'devolutiva',
} 


for contrib in df['id_contribuicao'].unique():
    df_contrib = df[df['id_contribuicao']==contrib]
    checar_qtds(df_contrib, colunas_atributos_str)
    checar_qtds(df_contrib, colunas_atributos_int)

    dados_parsed_contrib = {
        nome: extract_first_value(df_contrib, coluna, str)
        for coluna, nome in colunas_atributos_str.items()
    }

    for coluna, nome in colunas_atributos_int.items():
        dados_parsed_contrib[nome] = extract_first_value(df_contrib, coluna, int)

    for coluna, nome in colunas_atributos_many.items():

        dados_parsed_contrib[nome] = list(df_contrib[coluna].values)

    subs_copy = copy.deepcopy(dados_parsed_contrib['subprefeitura'])
    for sub in subs_copy:

        if sub is None or sub.strip()=='':
            dados_parsed_contrib['subprefeitura'].remove(sub)
            continue

        if (',' in sub) or (' e ' in sub) or (';' in sub):
            new_subs = re.split(',| e |;', sub)
            dados_parsed_contrib['subprefeitura'].remove(sub)
            for new_sub in new_subs:
                new_sub = new_sub.strip()
                if new_sub not in dados_parsed_contrib['subprefeitura']:
                    dados_parsed_contrib['subprefeitura'].append(new_sub)
                

        
    dados_parsed_contrib['devolutivas'] = []
    for devolutiva in df_contrib['devolutiva'].unique():
        df_devolutiva = df_contrib[df_contrib['devolutiva'] == devolutiva]

        checar_qtds(df_devolutiva, colunas_devolutiva)
        orgao_null = not checar_not_null(df_devolutiva, 'orgao')
        if orgao_null:
            continue
        dados_parsed_devolutiva = {
            nome: df_devolutiva[coluna].iloc[0]
            for coluna, nome in colunas_devolutiva.items()
        }

        dados_parsed_contrib['devolutivas'].append(dados_parsed_devolutiva)

    dados_parsed.append(dados_parsed_contrib)


In [18]:
df.columns

Index(['id_dev', 'id_cpma', 'id_participe_mais', 'canal_pad', 'tipo_origem',
       'municipe_pad', 'titulo', 'resumo', 'conteudo_ajustado', 'tema_pad',
       'orgao', 'devolutiva', 'apoios', 'total_comentarios',
       'subprefeitura_pad', 'id_contribuicao', 'is_fala'],
      dtype='object')

In [19]:
diagnostico

{}

In [20]:
dados_parsed

[{'id_contribuicao': 'pro_03808',
  'canal': 'Participe Mais',
  'tipo_origem': 'proposta',
  'municipe': 'Ricardo Souza',
  'titulo': 'Segunda Entrada para o Jardim Keralux.',
  'resumo': 'Agilizar um segundo acesso ao jardim Keralux, por estrada construída na parte interna da estação de trem Comendador Ermelino.',
  'conteudo': 'Minha proposta visa a criação de um segundo acesso para o Jardim Keralux, que não seja tão dispendioso financeiramente falando e seja uma ação rapida e com entrega agil. Pois a Rua Dom Jaime Echevarria, é uma opção, pois ela ja esta toda asfaltada e com iluminação publica e ela leva direto para a estação Comendado Ermelino Matarazzo, porem se em comum acordo com o Estado de São Paulo, derrubar o muro que passa pela mesma rua, porem agora dentro da massa falida da PanQuimica, Viscofam e ai sim se derruba o segundo muro e ja faz a interligação com a a rua principal do jardim Keralux que é a Arlindo Betio. Uma proposta rapida, que não tera necessidade de gastos 

In [21]:
diagnostico

{}

In [22]:
diagnostico_to_df = []
for contrib, msgs in diagnostico.items():
    for msg in msgs:
        diagnostico_to_df.append({
            'id_contribuicao': contrib,
            'mensagem': msg
        })

df_diagnostico = pd.DataFrame(diagnostico_to_df)
df_diagnostico.head()

""


In [23]:
df_diagnostico.to_csv('diagnostico.csv', index=False, sep=';')

In [24]:
path_json = os.path.join(path_dados_modulo, 'devolutivas.json')

with open(path_json, 'w', encoding='utf-8') as f:
    json.dump(dados_parsed, f, indent=4, ensure_ascii=False)